In [1]:
%%capture
%pip install -q "anthropic>=0.91.0" python-dotenv slack_bolt requests markdown-to-mrkdwn

In [2]:
import io
import os
import threading
from getpass import getpass
from pathlib import Path

import requests
from anthropic import Anthropic
from dotenv import load_dotenv, set_key
from markdown_to_mrkdwn import SlackMarkdownConverter
from slack_bolt import App
from slack_bolt.adapter.socket_mode import SocketModeHandler
 
load_dotenv(override=True)

# Prompt for Slack tokens on first run and save them to .env.
for key in ("SLACK_BOT_TOKEN", "SLACK_APP_TOKEN"):
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")
        set_key(".env", key, os.environ[key])
 
client = Anthropic()
app = App(token=os.environ["SLACK_BOT_TOKEN"])
 
for key in ("ANALYST_ENV_ID", "ANALYST_AGENT_ID", "ANALYST_AGENT_VERSION"):
    if not os.environ.get(key):
        raise RuntimeError(f"{key} not set. Run data_analyst_agent.ipynb first.")
 
# Set these from the IDs saved by the data analyst notebook. Reusing
# the agent and environment avoids re-provisioning on every bot restart.
ANALYST_AGENT = {
    "id": os.environ["ANALYST_AGENT_ID"],
    "version": int(os.environ["ANALYST_AGENT_VERSION"]),
}
ANALYST_ENV_ID = os.environ["ANALYST_ENV_ID"]

# thread_ts -> session_id, so follow-ups land in the same session.
# Sessions stay open for replies. In production, persist this and
# archive sessions when threads go stale.
thread_sessions: dict[str, str] = {}

mrkdwn = SlackMarkdownConverter()

## Start session when bot is mentioned

In [3]:
@app.event("app_mention")
def on_mention(event, say, ack):
    ack()
    print("APP MENTION RECEIVED:", event)
    say(text="On it. Analyzing now.", thread_ts=event.get("thread_ts") or event["ts"])
    channel = event["channel"]
    thread_ts = event.get("thread_ts") or event["ts"]
    # Mention text arrives as "<@BOTID> question"; drop the mention prefix.
    question = event["text"].split(">", 1)[-1].strip()
    slack_file = (event.get("files") or [None])[0]
     # Run the slow work in a background thread so this handler
    # returns within Slack's 3s limit.
    threading.Thread(target=start_analysis, args=(channel, thread_ts, question, slack_file)).start()


def start_analysis(channel: str, thread_ts: str, question: str, slack_file: dict | None) -> None:
    try:
        # If the mention had a file attached, pull it from Slack and
        # re-upload to the Anthropic Files API so the session can mount it.
        resources = []
        if slack_file:
            resp = requests.get(
                slack_file["url_private"],
                headers={"Authorization": f"Bearer {app.client.token}"},
                timeout=30,
            )
            resp.raise_for_status()
            mime = slack_file.get("mimetype", "text/csv")
            uploaded = client.beta.files.upload(
                file=(slack_file["name"], io.BytesIO(resp.content), mime)
            )
            mount = "/mnt/session/uploads/data.csv"
            resources.append({"type": "file", "file_id": uploaded.id, "mount_path": mount})
            question += f"\n\nThe data is mounted at {mount}."
 
        # One session per Slack thread. Store the thread coordinates in
        # metadata so anyone reading the event stream knows where to reply.
        session = client.beta.sessions.create(
            environment_id=ANALYST_ENV_ID,
            agent={"type": "agent", **ANALYST_AGENT},
            resources=resources,
            # Titles are capped at 80 chars and can't contain Unicode
            # control/format characters (Slack sometimes inserts them).
            title="".join(c for c in question if c.isprintable())[:80],
            metadata={"slack_channel": channel, "slack_thread_ts": thread_ts},
        )
        thread_sessions[thread_ts] = session.id
 
        # Send the question as a user.message event. The agent starts
        # working immediately; relay_stream posts its progress to the thread.
        client.beta.sessions.events.send(
            session.id,
            events=[{"type": "user.message", "content": [{"type": "text", "text": question}]}],
        )
        relay_stream(session.id, channel, thread_ts)
    except Exception as e:
        app.client.chat_postMessage(
            channel=channel, thread_ts=thread_ts, text=f"Analysis failed: {type(e).__name__}: {e}"
        )

## Relay progress and results to thread 

In [4]:
def relay_stream(session_id: str, channel: str, thread_ts: str) -> None:
    summary = ""
    posted_progress = False
    for ev in client.beta.sessions.events.stream(session_id):
        t = ev.type
        if t == "agent.message":
            # Keep the latest text block; it becomes the final summary.
            for b in ev.content:
                if b.type == "text" and b.text.strip():
                    summary = b.text
        elif t == "agent.tool_use" and not posted_progress:
            # Post a one-time progress update when the agent starts
            # running commands.
            app.client.chat_postMessage(
                channel=channel, thread_ts=thread_ts, text="Running analysis..."
            )
            posted_progress = True
        elif t == "session.status_idle":
            break
        elif t == "session.status_terminated":
            trace = f"https://platform.claude.com/sessions/{session_id}"
            app.client.chat_postMessage(
                channel=channel,
                thread_ts=thread_ts,
                text=f"Session terminated unexpectedly. Trace: {trace}",
            )
            return

    # Turn is done. Post the summary, then upload any generated files.
    if summary:
        text = mrkdwn.convert(summary)
        if len(text) > 3900:  # Slack text limit ~4000 chars
            text = text[:3900] + "\n_(truncated)_"
        app.client.chat_postMessage(channel=channel, thread_ts=thread_ts, text=text)
        
    outputs = client.beta.files.list(scope_id=session_id, betas=["managed-agents-2026-04-01"])
    Path("generated_scripts").mkdir(exist_ok=True)

    for f in outputs.data:
        if not f.downloadable:
            continue

        content = client.beta.files.download(f.id).read()

        # Save generated Python scripts locally
        if f.filename.endswith(".py"):
            Path("generated_scripts", f.filename).write_bytes(content)

        # Upload all generated files back to Slack
        app.client.files_upload_v2(
            channel=channel,
            thread_ts=thread_ts,
            filename=f.filename,
            content=content
        )
    
    for f in outputs.data:
        if not f.downloadable:
            continue
        content = client.beta.files.download(f.id).read()
        app.client.files_upload_v2(
            channel=channel, thread_ts=thread_ts, filename=f.filename, content=content
        )

## Handle follow ups in same session

In [5]:
def continue_session(session_id: str, channel: str, thread_ts: str, text: str) -> None:
    try:
        client.beta.sessions.events.send(
            session_id,
            events=[{"type": "user.message", "content": [{"type": "text", "text": text}]}],
        )
        relay_stream(session_id, channel, thread_ts)
    except Exception as e:
        app.client.chat_postMessage(
            channel=channel, thread_ts=thread_ts, text=f"Analysis failed: {type(e).__name__}: {e}"
        )
 

@app.event("message")
def on_thread_reply(event, ack):
    ack()
    thread_ts = event.get("thread_ts")
    # Only handle human replies in a thread where we already started
    # a session. Skip edits/deletes and other message subtypes.
    if event.get("subtype"):
        return
    if not thread_ts or event.get("bot_id") or thread_ts not in thread_sessions:
        return
    threading.Thread(
        target=continue_session,
        args=(thread_sessions[thread_ts], event["channel"], thread_ts, event["text"]),
    ).start()

## Run the bot

In [ ]:
SocketModeHandler(app, os.environ["SLACK_APP_TOKEN"]).start()

⚡️ Bolt app is running!
APP MENTION RECEIVED: {'type': 'app_mention', 'text': '<@U0BA981CSHM> could you identify the different customer profiles in this banking data and identify groups with potentially high credit risk?', 'files': [{'id': 'F0BATUGF8LF', 'created': 1781711085, 'timestamp': 1781711085, 'mimetype': 'text/csv', 'filetype': 'csv', 'pretty_type': 'CSV', 'user': 'U0793J0QEDN', 'user_team': 'TUARCCJD8', 'editable': True, 'size': 580598, 'mode': 'snippet', 'is_external': False, 'external_type': '', 'is_public': False, 'public_url_shared': False, 'display_as_bot': False, 'username': '', 'name': 'Banking.csv', 'title': 'Banking.csv', 'is_modified_by_ai': False, 'url_private': 'https://files.slack.com/files-pri/TUARCCJD8-F0BATUGF8LF/banking.csv', 'url_private_download': 'https://files.slack.com/files-pri/TUARCCJD8-F0BATUGF8LF/download/banking.csv', 'permalink': 'https://aberdeenadv.slack.com/files/U0BA981CSHM/F0BATUGF8LF/banking.csv', 'permalink_public': 'https://slack-files.com/

APP MENTION RECEIVED: {'type': 'app_mention', 'text': '<@U0BA981CSHM> could you identify the different customer profiles in this banking data and identify groups with potentially high credit risk?', 'files': [{'id': 'F0BBD73KKAQ', 'created': 1781712077, 'timestamp': 1781712077, 'mimetype': 'text/csv', 'filetype': 'csv', 'pretty_type': 'CSV', 'user': 'U0793J0QEDN', 'user_team': 'TUARCCJD8', 'editable': True, 'size': 580598, 'mode': 'snippet', 'is_external': False, 'external_type': '', 'is_public': False, 'public_url_shared': False, 'display_as_bot': False, 'username': '', 'name': 'Banking.csv', 'title': 'Banking.csv', 'is_modified_by_ai': False, 'url_private': 'https://files.slack.com/files-pri/TUARCCJD8-F0BBD73KKAQ/banking.csv', 'url_private_download': 'https://files.slack.com/files-pri/TUARCCJD8-F0BBD73KKAQ/download/banking.csv', 'permalink': 'https://aberdeenadv.slack.com/files/U0BA981CSHM/F0BBD73KKAQ/banking.csv', 'permalink_public': 'https://slack-files.com/TUARCCJD8-F0BBD73KKAQ-d7